# Fraud Detection Project - Part 1: Data Exploration

## What we'll do in this notebook:
1. Load the credit card transaction data
2. Understand what each column means
3. See how EXTREME the class imbalance is
4. Analyze the Time column
5. Look at transaction amounts
6. Visualize the problem

**Goal:** Understand the data before building any models

In [ ]:
# Import libraries
import pandas as pd  # For loading and working with data
import numpy as np   # For numerical operations
import matplotlib.pyplot as plt  # For creating plots
import seaborn as sns  # For prettier plots

# Make plots appear in the notebook
%matplotlib inline

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded successfully!")

## Step 1: Load the Data

The dataset is a CSV file (like an Excel spreadsheet but simpler).
Each row = one credit card transaction.
Each column = information about that transaction.

In [ ]:
# Load the dataset
# pd.read_csv() reads a CSV file and converts it to a DataFrame (pandas table)
df = pd.read_csv('../data/creditcard.csv')

print(f"Dataset loaded!")
print(f"Number of transactions: {len(df):,}")  # :, adds commas to big numbers
print(f"Number of features (columns): {df.shape[1]}")
print(f"\nFirst few rows:")
df.head()  # Shows first 5 rows

## Step 2: Understand the Columns

**What each column means:**

- **Time**: Seconds elapsed between this transaction and the first transaction in the dataset
  - This tells us WHEN the transaction happened
  - We'll use this to split data by time (train on old, test on new)

- **V1, V2, ..., V28**: These are the main features
  - They've been transformed using PCA (Principal Component Analysis)
  - PCA was done for privacy - we can't see the original features
  - But they still contain all the important patterns
  - Think of them as "anonymous fingerprints" of each transaction

- **Amount**: Transaction amount in dollars (or euros, the dataset doesn't specify)
  - This is the ONLY feature we can actually interpret
  - Higher amounts might be more suspicious

- **Class**: The target variable (what we're trying to predict)
  - 0 = Legitimate transaction (normal)
  - 1 = Fraudulent transaction (fraud!)
  - This is what our model will learn to predict

In [ ]:
# Check column names and types
print("Column information:")
print(df.info())

print("\n" + "="*50)
print("Basic statistics:")
df.describe()

## Step 3: The EXTREME Class Imbalance

This is the most important thing to understand about this dataset.
Fraud is VERY rare in real life, and our data reflects that.

In [ ]:
# Count how many frauds vs legitimate transactions
class_counts = df['Class'].value_counts()
print("Transaction counts:")
print(f"Legitimate (0): {class_counts[0]:,}")
print(f"Fraud (1):      {class_counts[1]:,}")

# Calculate fraud percentage
fraud_percentage = (class_counts[1] / len(df)) * 100
print(f"\nFraud rate: {fraud_percentage:.3f}%")
print(f"That's only {fraud_percentage:.3f} frauds per 100 transactions!")
print(f"Or about 1 fraud in every {int(100/fraud_percentage)} transactions")

print("\n" + "!"*50)
print("WHY THIS MATTERS:")
print("A dumb model that always predicts 'legitimate' would be")
print(f"correct {100-fraud_percentage:.2f}% of the time!")
print("But it would catch ZERO frauds.")
print("That's why accuracy is a terrible metric here.")
print("!"*50)

In [ ]:
# Visualize the imbalance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax1 = axes[0]
class_counts.plot(kind='bar', ax=ax1, color=['green', 'red'])
ax1.set_title('Class Distribution (Absolute Counts)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Class (0=Legit, 1=Fraud)')
ax1.set_ylabel('Number of Transactions')
ax1.set_xticklabels(['Legitimate', 'Fraud'], rotation=0)

# Add count labels on bars
for i, v in enumerate(class_counts):
    ax1.text(i, v + 5000, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
ax2 = axes[1]
colors = ['green', 'red']
explode = (0, 0.1)  # Explode the fraud slice
ax2.pie(class_counts, labels=['Legitimate', 'Fraud'], autopct='%1.3f%%',
        colors=colors, explode=explode, startangle=90)
ax2.set_title('Class Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/class_imbalance.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nPlot saved to: results/figures/class_imbalance.png")

## Step 4: Analyze the Time Column

Time is in seconds. Let's convert it to hours to understand the data span.

In [ ]:
# Time statistics
print("Time range (in seconds):")
print(f"Minimum: {df['Time'].min():,} seconds")
print(f"Maximum: {df['Time'].max():,} seconds")
print(f"Total span: {df['Time'].max():,} seconds")

# Convert to hours
time_hours = df['Time'].max() / 3600
time_days = time_hours / 24
print(f"\nIn human terms:")
print(f"  {time_hours:.1f} hours")
print(f"  {time_days:.1f} days")

print(f"\nThis dataset covers about 2 days of transactions.")
print(f"That's one limitation - we can't see true long-term drift.")

In [ ]:
# Plot transaction volume over time
# Create time bins (every 2 hours = 7200 seconds)
df['TimeHours'] = df['Time'] / 3600  # Convert to hours

plt.figure(figsize=(14, 6))

# All transactions
plt.subplot(2, 1, 1)
plt.hist(df['TimeHours'], bins=48, color='blue', alpha=0.7, edgecolor='black')
plt.title('Transaction Volume Over Time (All Transactions)', fontsize=14, fontweight='bold')
plt.xlabel('Time (hours)')
plt.ylabel('Number of Transactions')
plt.grid(axis='y', alpha=0.3)

# Fraud transactions only
fraud_df = df[df['Class'] == 1]
plt.subplot(2, 1, 2)
plt.hist(fraud_df['TimeHours'], bins=48, color='red', alpha=0.7, edgecolor='black')
plt.title('Fraud Transaction Volume Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Time (hours)')
plt.ylabel('Number of Frauds')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/time_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print("Notice: Transaction patterns change over time.")
print("This is why we need drift monitoring!")

## Step 5: Analyze Transaction Amounts

Are fraudulent transactions typically larger or smaller?

In [ ]:
# Amount statistics by class
print("Transaction Amount Statistics:\n")
print("LEGITIMATE transactions:")
legit_amounts = df[df['Class'] == 0]['Amount']
print(f"  Mean: ${legit_amounts.mean():.2f}")
print(f"  Median: ${legit_amounts.median():.2f}")
print(f"  Max: ${legit_amounts.max():.2f}")

print("\nFRAUD transactions:")
fraud_amounts = df[df['Class'] == 1]['Amount']
print(f"  Mean: ${fraud_amounts.mean():.2f}")
print(f"  Median: ${fraud_amounts.median():.2f}")
print(f"  Max: ${fraud_amounts.max():.2f}")

print("\n" + "-"*50)
print("Key insight: Fraud amounts are actually SMALLER on average!")
print("This shows Amount alone won't catch fraud - we need all V1-V28 features.")

In [ ]:
# Visualize amount distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
ax1 = axes[0]
df.boxplot(column='Amount', by='Class', ax=ax1)
ax1.set_title('Amount Distribution by Class', fontsize=14, fontweight='bold')
ax1.set_xlabel('Class (0=Legit, 1=Fraud)')
ax1.set_ylabel('Amount ($)')
plt.sca(ax1)
plt.xticks([1, 2], ['Legitimate', 'Fraud'])

# Histogram with log scale
ax2 = axes[1]
ax2.hist(legit_amounts[legit_amounts < 500], bins=50, alpha=0.7, 
         label='Legitimate', color='green', edgecolor='black')
ax2.hist(fraud_amounts[fraud_amounts < 500], bins=50, alpha=0.7, 
         label='Fraud', color='red', edgecolor='black')
ax2.set_title('Amount Distribution (capped at $500)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Amount ($)')
ax2.set_ylabel('Frequency')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/amount_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

## Step 6: Check for Missing Values

Missing data can break ML models. Let's make sure we don't have any.

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values! Perfect.")

# Check for any weird values
print("\nData types:")
print(df.dtypes.value_counts())
print("\nAll columns are numeric - good for ML!")

## Step 7: Look at Feature Correlations

Some V features might be correlated with Class (fraud).
High correlation = that feature is useful for detecting fraud!

In [ ]:
# Calculate correlation with Class
correlations = df.corr()['Class'].drop('Class').sort_values(key=abs, ascending=False)

print("Top 10 features most correlated with Fraud:\n")
for i, (feature, corr) in enumerate(correlations.head(10).items(), 1):
    direction = "positive" if corr > 0 else "negative"
    print(f"{i:2d}. {feature:6s}: {corr:+.3f} ({direction} correlation)")

print("\nPositive correlation: higher value = more likely fraud")
print("Negative correlation: lower value = more likely fraud")

In [ ]:
# Plot top correlations
plt.figure(figsize=(12, 6))
top_features = correlations.head(15)
colors = ['red' if x < 0 else 'green' for x in top_features.values]
plt.barh(range(len(top_features)), top_features.values, color=colors)
plt.yticks(range(len(top_features)), top_features.index)
plt.xlabel('Correlation with Class (Fraud)')
plt.title('Top 15 Features Correlated with Fraud', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/feature_correlations.png', dpi=100, bbox_inches='tight')
plt.show()

print("These features will be most important for our models!")

## Summary: What We Learned

### Key Findings:

1. **Extreme Class Imbalance**
   - Only 0.173% fraud (492 out of 284,807)
   - Accuracy is useless - a model predicting "all legit" gets 99.827% accuracy but catches zero frauds
   - We MUST use Precision-Recall metrics instead

2. **Time Span**
   - Data covers ~2 days (48 hours)
   - Transaction patterns vary over time
   - We'll use time-based split (not random!) to simulate real deployment

3. **Transaction Amounts**
   - Fraud amounts are actually SMALLER on average
   - Amount alone won't catch fraud
   - Need the V1-V28 features (which we can't interpret due to PCA)

4. **No Missing Data**
   - Clean dataset, ready for modeling
   - All features are numeric

5. **Feature Importance**
   - Some V features have strong correlations with fraud
   - These will be the most predictive features

### Next Steps:
- Implement time-based data split
- Handle class imbalance (SMOTE, class weights)
- Train models
- Set up drift monitoring

In [ ]:
# Save a summary
summary = {
    'total_transactions': len(df),
    'fraud_count': class_counts[1],
    'fraud_percentage': fraud_percentage,
    'time_span_hours': time_hours,
    'time_span_days': time_days,
    'mean_amount_legit': legit_amounts.mean(),
    'mean_amount_fraud': fraud_amounts.mean(),
    'top_correlated_features': correlations.head(10).to_dict()
}

import json
with open('../results/exploration_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\nExploration complete!")
print("Summary saved to: results/exploration_summary.json")
print("\nYou now understand the data. Ready for modeling!")